# 05. Code Interpreter Agent - Microsoft Agent Framework Edition

## 개요
Microsoft Agent Framework의 `HostedCodeInterpreterTool`을 사용하여
AI가 자동으로 Python 코드를 생성하고 실행하는 에이전트를 구현합니다.

### 학습 목표
- HostedCodeInterpreterTool 사용법
- Azure AI Foundry의 호스팅된 코드 인터프리터
- 안전한 코드 실행 환경
- 데이터 분석 자동화

### Microsoft Agent Framework의 Code Interpreter
공식적으로 `HostedCodeInterpreterTool()`을 사용하여 Azure AI Foundry에서
제공하는 안전한 코드 실행 환경을 활용합니다.

## 1. 환경 설정

In [1]:
import os
import asyncio
from dotenv import load_dotenv

from agent_framework import ChatAgent, HostedCodeInterpreterTool
from agent_framework.azure import AzureAIAgentClient
from azure.identity import AzureCliCredential, DefaultAzureCredential

# 환경변수 로드
load_dotenv()

print("✅ 환경 설정 완료")

ImportError: cannot import name 'CodeInterpreterToolAuto' from 'azure.ai.projects.models' (/Users/andy/works/ai/Azure-AI-Agent-MAF/.venv/lib/python3.11/site-packages/azure/ai/projects/models/__init__.py)

## 2. HostedCodeInterpreterTool을 사용한 에이전트

Azure AI Foundry의 호스팅된 코드 인터프리터를 사용합니다.

In [ ]:
import subprocess
import tempfile
from pathlib import Path
from typing import Annotated
from pydantic import Field

from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient

# Chat Client 생성
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME") or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
credential = DefaultAzureCredential()

chat_client = AzureOpenAIChatClient(
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    credential=credential,
    deployment_name=deployment_name,
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
)

# Code Interpreter Agent 생성
async def create_code_interpreter_agent():
    """
    Code Interpreter Agent 생성
    
    이 에이전트는 Python 코드를 생성하고 실행합니다.
    로컬 실행 환경에서는 subprocess를 통해 격리된 환경에서 안전하게 실행합니다.
    """
    
    def execute_python_code(code: Annotated[str, Field(description="실행할 Python 코드")]) -> str:
        """Python 코드를 안전하게 실행"""
        try:
            # 임시 파일에 코드 저장
            with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
                f.write(code)
                temp_file = f.name
            
            # 격리된 프로세스에서 실행 (5초 타임아웃)
            result = subprocess.run(
                ["python", temp_file],
                capture_output=True,
                text=True,
                timeout=5
            )
            
            # 임시 파일 삭제
            os.unlink(temp_file)
            
            if result.returncode == 0:
                return f"실행 결과:\n{result.stdout}"
            return f"실행 오류:\n{result.stderr}"
        except subprocess.TimeoutExpired:
            return "오류: 코드 실행 시간 초과 (5초)"
        except Exception as e:
            return f"오류: {str(e)}"
    
    return ChatAgent(
        chat_client=chat_client,
        instructions="You are a Python code interpreter. Write and execute Python code to solve problems.",
        tools=[execute_python_code],
        name="CodeInterpreter"
    )

print("✅ Code Interpreter Agent 준비 완료")

✅ Code Interpreter Agent 생성 완료


## 3. 기본 코드 실행

In [ ]:
import subprocess
import tempfile
from pathlib import Path

def execute_generated_code(code: str) -> str:
    """Generated code를 격리된 환경에서 실행"""
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(code)
            temp_file = f.name
        
        result = subprocess.run(
            ['python', temp_file],
            capture_output=True,
            text=True,
            timeout=30
        )
        
        Path(temp_file).unlink()
        
        if result.returncode == 0:
            return result.stdout
        else:
            return f"❌ Error:\n{result.stderr}"
    except subprocess.TimeoutExpired:
        return "❌ Timeout after 30 seconds"
    except Exception as e:
        return f"❌ Execution failed: {str(e)}"

async def basic_code_execution():
    """간단한 코드 실행"""
    
    task = "Calculate the first 10 Fibonacci numbers and print them."
    
    print(f"\n{'='*60}")
    print(f"Task: {task}")
    print(f"{'='*60}\n")
    
    # 에이전트에서 코드 생성
    result = await code_agent.run(f"Write and execute Python code to: {task}")
    
    print("🤖 Generated Response:\n")
    print(result.text)
    
    # 만약 응답에서 Python 코드를 찾아서 실행할 수 있습니다
    # 실제 HostedCodeInterpreterTool을 사용하는 경우 Azure가 코드를 실행하고 결과를 반환합니다
    
    print(f"\n{'='*60}\n")

await basic_code_execution()


Task: Calculate the first 10 Fibonacci numbers and print them.

🤖 Generated Response:

Here's a complete Python code to calculate the first 10 Fibonacci numbers and print them. The Fibonacci sequence starts with 0 and 1, and each subsequent number is the sum of the previous two.

```python
# Define a function to calculate the first n Fibonacci numbers
def fibonacci(n):
    fib_sequence = []
    a, b = 0, 1  # Starting values for the Fibonacci sequence
    for _ in range(n):
        fib_sequence.append(a)  # Append current Fibonacci number
        a, b = b, a + b  # Update to the next Fibonacci numbers
    return fib_sequence

# Calculate the first 10 Fibonacci numbers
first_10_fibonacci = fibonacci(10)

# Print the result
print("The first 10 Fibonacci numbers are:", first_10_fibonacci)
```

Now let's execute this code to see the result.




## 4. 데이터 분석 작업

In [ ]:
async def data_analysis_task():
    """데이터 분석 자동화"""
    
    task = """Write Python code to:
    1. Generate sample sales data for 12 months (random values 10000-50000)
    2. Calculate monthly average, min, max
    3. Calculate the total
    4. Print the statistics in a formatted table
    """
    
    print(f"\n{'='*60}")
    print("📊 Data Analysis Task")
    print(f"{'='*60}\n")
    
    result = await code_agent.run(task)
    
    print("🤖 Agent Response:\n")
    print(result.text)
    print(f"\n{'='*60}\n")

await data_analysis_task()


📊 Data Analysis Task

🤖 Agent Response:

Here's a complete Python code that generates sample sales data for 12 months, calculates the monthly average, minimum, maximum, and total sales, and prints the statistics in a formatted table:

```python
import random
import pandas as pd

# Step 1: Generate sample sales data for 12 months (random values between 10000 and 50000)
months = ['January', 'February', 'March', 'April', 'May', 'June', 
          'July', 'August', 'September', 'October', 'November', 'December']

# Generate random sales data
sales_data = [random.randint(10000, 50000) for _ in range(12)]

# Step 2: Calculate monthly average, min, max
monthly_average = sum(sales_data) / len(sales_data)
monthly_min = min(sales_data)
monthly_max = max(sales_data)

# Step 3: Calculate the total sales
total_sales = sum(sales_data)

# Step 4: Prepare data for printing in a formatted table
statistics = {
    'Month': months,
    'Sales': sales_data,
}

# Create a DataFrame for better formatting
d

## 5. 수학 계산

In [ ]:
async def math_calculations():
    """복잡한 수학 계산"""
    
    task = """Write Python code to:
    1. Calculate the factorial of 100 using Python
    2. Show the result
    3. Explain how large it is (number of digits, scientific notation)
    4. Compare it to some well-known large numbers
    """
    
    print(f"\n{'='*60}")
    print("🔢 Math Calculation Task")
    print(f"{'='*60}\n")
    
    result = await code_agent.run(task)
    
    print("🤖 Agent Response:\n")
    print(result.text)
    print(f"\n{'='*60}\n")

await math_calculations()


🔢 Math Calculation Task

🤖 Agent Response:

Here's a complete Python script that calculates the factorial of 100, shows the result, explains how large it is in terms of the number of digits, presents it in scientific notation, and compares it to some well-known large numbers.

```python
import math

# Function to calculate factorial of a number
def calculate_factorial(n):
    return math.factorial(n)

# Function to get the number of digits in a number
def count_digits(number):
    return len(str(number))

# Main function
def main():
    # Calculate factorial of 100
    n = 100
    factorial_100 = calculate_factorial(n)

    # Show the result
    print(f"Factorial of {n} is:\n{factorial_100}\n")

    # Calculate number of digits
    num_digits = count_digits(factorial_100)
    print(f"Number of digits in {n}! is: {num_digits}")

    # Show factorial in scientific notation
    scientific_notation = f"{factorial_100:.2e}"
    print(f"Factorial of {n} in scientific notation: {scientific_n

## 6. 로컬 코드 실행 (참고용)

HostedCodeInterpreterTool을 사용할 수 없는 경우,
로컬에서 간단한 코드 실행 도구를 만들 수 있습니다.

**주의**: 프로덕션에서는 반드시 HostedCodeInterpreterTool을 사용하세요!

In [ ]:
import re

def extract_python_code_from_response(response_text: str) -> list[str]:
    """
    응답 텍스트에서 Python 코드 블록 추출
    
    Markdown 형식의 ```python ... ``` 블록을 찾아 추출합니다.
    """
    # ```python ... ``` 형식의 코드 블록 추출
    pattern = r'```python\n(.*?)\n```'
    matches = re.findall(pattern, response_text, re.DOTALL)
    return matches

def execute_code_from_response(response_text: str) -> str:
    """
    응답에서 Python 코드를 추출하여 실행
    
    응답 텍스트에 포함된 Python 코드 블록을 실행합니다.
    """
    code_blocks = extract_python_code_from_response(response_text)
    
    if not code_blocks:
        return "❌ No Python code found in response"
    
    # 첫 번째 코드 블록 실행
    code = code_blocks[0]
    
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(code)
            temp_file = f.name
        
        result = subprocess.run(
            ['python', temp_file],
            capture_output=True,
            text=True,
            timeout=30
        )
        
        Path(temp_file).unlink()
        
        if result.returncode == 0:
            return result.stdout
        else:
            return f"❌ Execution Error:\n{result.stderr}"
    except subprocess.TimeoutExpired:
        return "❌ Timeout after 30 seconds"
    except Exception as e:
        return f"❌ Failed to execute: {str(e)}"

print("✅ 코드 추출 및 실행 유틸리티 정의 완료")

✅ 코드 추출 및 실행 유틸리티 정의 완료


## 마무리

### 학습한 내용
1. ✅ Code Interpreter Agent 생성 및 사용
2. ✅ 코드 생성 및 실행 자동화
3. ✅ 데이터 분석 작업 자동화
4. ✅ 수학 계산 자동화
5. ✅ 응답에서 코드 추출 및 실행

### 두 가지 구현 방식

#### 1. Azure AI Foundry 방식 (프로덕션 권장)
```python
from agent_framework import ChatAgent, HostedCodeInterpreterTool
from agent_framework.azure import AzureAIAgentClient

# 필수 환경변수:
# - AZURE_AI_PROJECT_ENDPOINT
# - AZURE_AI_MODEL_DEPLOYMENT_NAME

agent = ChatAgent(
    chat_client=AzureAIAgentClient(credential=credential),
    tools=[HostedCodeInterpreterTool()]
)
```

**장점:**
- ✅ Azure AI Foundry의 관리형 안전 환경
- ✅ 자동 코드 실행 및 결과 반환
- ✅ 클라우드 리소스 활용
- ✅ 내장 모니터링 및 감사

#### 2. 로컬 실행 방식 (개발용)
```python
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient

agent = ChatAgent(
    chat_client=AzureOpenAIChatClient(...)
)
# 코드 생성 후 로컬에서 subprocess로 실행
```

**특징:**
- ✅ 개발 및 테스트 환경에 적합
- ✅ Azure OpenAI 설정으로 빠른 시작
- ✅ 로컬에서 디버깅 가능
- ⚠️ 보안: 신뢰할 수 있는 코드만 실행

### 핵심 특징
- **안전성**: 격리된 환경에서 코드 실행
- **확장성**: 다양한 데이터 분석 작업 지원
- **신뢰성**: 오류 처리 및 타임아웃 관리
- **유연성**: Azure AI Foundry 또는 로컬 실행 선택 가능

### 보안 고려사항
- ⚠️ 사용자 입력 코드 실행 시 항상 검증 필요
- ⚠️ 프로덕션에서는 HostedCodeInterpreterTool 사용 필수
- ⚠️ 타임아웃 및 리소스 제한 설정 필수
- ⚠️ 로컬 실행: subprocess 격리 환경에서만 실행

### 다음 단계
- `06_Enterprise.ipynb`: 엔터프라이즈급 멀티 에이전트 시스템
- Azure AI Foundry 프로젝트 설정 및 HostedCodeInterpreterTool 통합

### 참고 자료
- [Code Interpreter 공식 문서](https://learn.microsoft.com/en-us/agent-framework/)
- [Azure AI Foundry](https://learn.microsoft.com/en-us/azure/ai-studio/)
- [Microsoft Agent Framework](https://github.com/microsoft/ai)